In [ ]:
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.isdir("smartstock-ai"):          # Colab/Kaggle: clone your repo first
    subprocess.run(["git", "clone", "https://github.com/<you>/smartstock-ai.git"], check=True); os.chdir("smartstock-ai")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "training/requirements.txt"], check=True)
ROOT = os.getcwd()
while not os.path.isdir(os.path.join(ROOT, "backend")) and ROOT != "/": ROOT = os.path.dirname(ROOT)
os.chdir(ROOT); sys.path[:0] = [os.path.join(ROOT, "backend"), os.path.join(ROOT, "training", "scripts")]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
RAW_OK = os.path.exists("data/raw/m5/sales_train_validation.csv")
if not RAW_OK:   # no real M5 download present -> generate M5-format synthetic data so the notebook runs anywhere
    subprocess.run([sys.executable, "training/scripts/make_synthetic_m5.py"], check=True)
if not os.path.exists("data/processed/panel.npz"):
    subprocess.run([sys.executable, "training/scripts/prepare_data.py"], check=True)
def ensure_trained():
    if not os.path.exists("training/outputs/metrics.json"):
        subprocess.run([sys.executable, "training/scripts/train.py"], check=True)

# 07 · Safety stock + PuLP placement
**All costs, lead times and capacities are simulation assumptions.**

In [ ]:
from app.models.optimizer import optimize_allocation
df = pd.DataFrame([["A","S1",0,10,.01,5,1], ["A","S2",0,10,.01,5,3]], columns=["item_id","store_id","current","target","hold","penalty","transport"])
r, info = optimize_allocation(df, {"A": 15}, {}); display(r[["store_id","current","target","ship","recommended","shortage"]]); print(info)
assert list(r.ship) == [10, 5]   # hand check: only 15 units for 20 needed -> fill the cheaper-to-ship store first

In [ ]:
from app.engine import Engine
e = Engine("backend/model_artifacts"); out = e.recommendation("FOODS_1_001", "CA_1"); print({k: out[k] for k in ("inventory", "action", "reason", "service_level")})
pd.DataFrame(out["placement"])